In [49]:
import warnings
for warn in [UserWarning, FutureWarning]: warnings.filterwarnings("ignore", category = warn)

import os
import io
import re
import sys
import math
import socket
import pickle
import random
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import tensorboard as tb
import torchinfo
import numpy as np
import pandas as pd
import polars as pl
import tabulate
import datasets
import torchvision as tv
import torchvision.utils as vutils
import torchvision.transforms as transforms
import PIL
import jupyterlab as jlab
import ipywidgets
import time
import matplotlib as mpl
import matplotlib.pyplot as plt
import tqdm
import logging

from dataclasses import dataclass
from typing import ClassVar

from pathlib import Path
from torch.optim import Adam
from torchvision.transforms import v2
from torchvision import transforms as T
from torch.nn.utils.rnn import pad_sequence
from torch.utils.tensorboard import SummaryWriter
from torch.utils.data import DataLoader
from datasets import load_dataset_builder, load_dataset, get_dataset_split_names, get_dataset_config_names, Dataset, Image, DatasetInfo
from datetime import datetime
from PIL import Image as PILImage, ImageFont, ImageDraw
from collections import Counter
from torchinfo import summary
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

from typing import List, Dict, Any, Tuple, Optional

### Задача -- классифицировать отзыв к фильму

### Классы -- положительный или отрицательный

#### Загрузка набора данных

In [50]:
train_ds = Dataset.from_json("./imdb_dataset/train.jsonl")
test_ds = Dataset.from_json("./imdb_dataset/test.jsonl")

#### Проверка атрибутов

In [51]:
print("class: ", train_ds[0]['label'])
print("text: ", train_ds[0]['text']) 

class:  0
text:  I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are few and

#### Настройка TensorBoard

In [52]:
log_dir = os.path.join(
    'runs','Word2Vec_imdb_' + datetime.now().strftime('%Y%m%d-%H%M%S') 
)
writer_tb = SummaryWriter(log_dir=log_dir)

#### Определение глобальных параметров

In [53]:
ROOT_DIR = os.path.join(".")

PATH_TO_MODEL = os.path.join(ROOT_DIR, "models") # Путь для сохранения моделей
PATH_TO_CACHE = os.path.join(ROOT_DIR, "imdb_dataset") # Путь для загрузки наборов данных

#### Определение гиперпараметров для обучения

In [54]:
EPOCHS = 10 # Количество эпох
BATCH_SIZE = 128 # Размер выборки (пакета)
LEARNING_RATE = 0.025 # Скорость обучения
EMBEDDINGS_DIM = 100 # Размерность векторного представления слов
SAMPLE = 1e-3 # Доля слов для обучения (для ускорения процесса обучения)
MIN_COUNT = 1
NEGATIVE = 5
WINDOW = 5
# Проверка наличия GPU
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


learning_hyperparams = {
    "Гиперпараметр": [
        "Количество эпох", "Размер выборки (пакета)", "Скорость обучения",
        "Размерность векторного представления слов", "Доля слов для обучения (для ускорения процесса обучения)",
        "Минимальная частота слова", "Количество негативных примеров", "Размер окна", "Устройство для обучения"
    ],
    "Значение": [
        str(EPOCHS), BATCH_SIZE, LEARNING_RATE, EMBEDDINGS_DIM, SAMPLE, MIN_COUNT, NEGATIVE, WINDOW, DEVICE
    ]
}

df_learning_hyperparams = pd.DataFrame(data = learning_hyperparams)
df_learning_hyperparams.index.name = "№"
df_learning_hyperparams.index += 1

display(df_learning_hyperparams)
# Отображение (TensorBoard)
writer_tb.add_text("Гиперпараметры для обучения", df_learning_hyperparams.to_markdown())

,Гиперпараметр,Значение
№,,
1,Количество эпох,10
2,Размер выборки (пакета),128
3,Скорость обучения,0.025
4,Размерность векторного представления слов,100
5,Доля слов для обучения (для ускорения процесса...,0.001
6,Минимальная частота слова,1
7,Количество негативных примеров,5
8,Размер окна,5
9,Устройство для обучения,cpu


#### Word2Vec model

In [55]:
class TorchWord2Vec(nn.Module):
    """Skip-gram word2vec model with negative sampling (PyTorch implementation)"""
    def __init__(self, vocab_size: int, embedding_dim: int):
        super().__init__()
        self.embeddings = nn.Embedding(vocab_size, embedding_dim)
        self.output_weights = nn.Embedding(vocab_size, embedding_dim)
        nn.init.uniform_(self.embeddings.weight, -0.5/embedding_dim, 0.5/embedding_dim)
        nn.init.uniform_(self.output_weights.weight, -0.5/embedding_dim, 0.5/embedding_dim)
    
    def forward(self, centers, contexts, negatives):
        center_vecs = self.embeddings(centers)
        context_vecs = self.output_weights(contexts)
        neg_vecs = self.output_weights(negatives)
        
        pos_score = torch.sum(center_vecs * context_vecs, dim=1)
        pos_loss = -torch.log(torch.sigmoid(pos_score) + 1e-12)
        
        neg_score = torch.bmm(neg_vecs, center_vecs.unsqueeze(2)).squeeze(2)
        neg_loss = -torch.sum(torch.log(torch.sigmoid(-neg_score) + 1e-12), dim=1)
        
        return torch.mean(pos_loss + neg_loss)


#### Класс обучения

In [ ]:
class Word2VecTrainer:
    def __init__(self,
                 sentences : List[List[str]],
                 batch_size : int = 16,
                 embedding_dim: int = 100,
                 window : int = 5,
                 min_count : int = 1,
                 negative : int = 5,
                 sample_thr : float = 1e-3,
                 epochs : int = 20,
                 lr : float = 1e-3):
        self.sentences = sentences
        self.batch_size = batch_size
        self.vector_size = embedding_dim
        self.window = window
        self.min_count = min_count
        self.negative = negative
        self.sample = sample_thr
        self.epochs = epochs
        self.lr = lr

        self._build_vocab()
        print(f"Vocab size: {len(self.vocab)}")

        self._build_unigram_table()        
        
    def _build_vocab(self):
        """
        Строит словарь слов на основе корпуса.
        
        Выполняет следующие действия:
        1. Подсчитывает частоту каждого слова во всех предложениях.
        2. Отфильтровывает слова с частотой ниже min_count.
        3. Назначает каждому оставшемуся слову уникальный индекс (от 0 до N-1).
        4. Рассчитывает вероятности субсэмплирования для частых слов (если включено).
        """
        word_counts = Counter()
        for sent in self.sentences:
            word_counts.update(sent)
        
        # keep only words with enough frequency and build contiguous indices
        frequent = [w for w, c in word_counts.items() if c >= self.min_count]
        self.vocab = {w: i for i, w in enumerate(frequent)}
        self.index2word = {i: w for w, i in self.vocab.items()}
        self.words_counts = {w: c for w, c in word_counts.items() if w in self.vocab}
        self.total_words = sum(self.words_counts.values())

        self.subsample_probs = {}
        if self.sample > 0:
            for word, count in self.words_counts.items():
                freq = count / self.total_words
                prob = 1 - math.sqrt(self.sample / freq)
                self.subsample_probs[word] = max(0, prob)


    def _build_unigram_table(self, table_size: int = 100_000_000):
        """
        Строит таблицу для эффективной негативной выборки (negative sampling).
        
        Таблица заполняется индексами слов с вероятностью, пропорциональной P(w)^0.75,
        где P(w) — частота слова в корпусе. Это ускоряет сэмплирование негативных примеров
        во время обучения, избегая дорогостоящих вычислений на каждом шаге.
        """
        vocab_words = list(self.vocab.keys())
        vocab_counts = np.array([self.words_counts[w] for w in vocab_words])
        unigram_probs = vocab_counts ** 0.75
        unigram_probs = unigram_probs / unigram_probs.sum()

        self.neg_table = np.random.choice(
            a=len(vocab_words),
            size=table_size,
            p=unigram_probs
        )
        self.neg_words = vocab_words

    def _sample_negative(self, center_idx: int, context_idx: int) -> int:
        """
        Сэмплирует индекс негативного слова для пары (центр, контекст).
        
        Возвращает индекс слова из словаря, которое:
        - не совпадает с центральным словом,
        - не совпадает с позитивным контекстным словом.
        
        Использует предварительно построенную таблицу негативной выборки для скорости.
        """
        while True:
            neg_idx = self.neg_table[random.randint(0, len(self.neg_table) - 1)]
            if neg_idx != center_idx and neg_idx != context_idx:
                return neg_idx
            
    def subsample(self, word: str) -> bool:
        """
        Определяет, должно ли слово быть исключено при субсэмплировании.
        
        Для частых слов возвращает False с вероятностью, зависящей от их частоты
        (чем чаще слово, тем выше шанс его пропустить). Это помогает уменьшить влияние
        стоп-слов и ускорить обучение.
        
        Возвращает:
            True — слово сохраняется для обучения,
            False — слово пропускается (субсэмплируется).
        """
        if word not in self.subsample_probs:
            return True
        return random.random() > self.subsample_probs[word]
    

    def _build_training_pairs(self):
        """
        Генерирует обучающие пары (центральное слово, контекстное слово).
        
        Для каждого предложения:
        1. Применяет субсэмплирование к словам.
        2. Для каждого слова как центра собирает контекстные слова в окне [-window, +window].
        3. Формирует список кортежей (индекс_центра, индекс_контекста).
        
        Результат сохраняется в self.pairs для последующего батчевого обучения.
        """
        pairs = []
        for sent in self.sentences:
            sent = [w for w in sent if w in self.vocab and self.subsample(w)]
            if len(sent) < 2:
                continue
            for i, center in enumerate(sent):
                window = random.randint(1, self.window)
                start = max(0, i - window)
                end = min(len(sent), i + window + 1)
                for j in range(start, end):
                    if i != j:
                        pairs.append((self.vocab[center], self.vocab[sent[j]]))
        self.pairs = pairs
        print(f"Generated {len(pairs):,} training pairs")

    def _generate_train_batch(self):
        """
        Формирует один обучающий батч с позитивными и негативными примерами.
        
        Для случайно выбранных пар (центр, контекст):
        1. Извлекает индексы центральных и контекстных слов.
        2. Для каждой пары генерирует `negative` негативных примеров.
        
        Возвращает:
            centers: тензор индексов центральных слов (размер batch_size),
            contexts: тензор индексов позитивных контекстных слов (размер batch_size),
            negatives: тензор индексов негативных слов (размер batch_size × negative).
        """
        batch_pairs = random.sample(self.pairs, self.batch_size)
        centers = [p[0] for p in batch_pairs]
        contexts = [p[1] for p in batch_pairs]
        negatives = [
            [self._sample_negative(center, context) for _ in range(self.negative)]
            for center, context in batch_pairs
        ]
        return (torch.LongTensor(centers),
                 torch.LongTensor(contexts),
                   torch.LongTensor(negatives))
    
    def train(self, writer=None):
        """
        Запускает полный цикл обучения модели Word2Vec методом негативной выборки.
        
        Этапы обучения:
        1. Инициализирует модель и оптимизатор (SGD).
        2. Формирует и логирует архитектуру модели через torchinfo.summary().
        3. Генерирует обучающие пары (центр, контекст).
        4. В каждой эпохе:
            - постепенно уменьшает скорость обучения (линейное затухание),
            - формирует батчи с негативными примерами,
            - вычисляет лосс и делает шаг оптимизации,
            - логирует средний лосс и текущую скорость обучения.
        5. После обучения извлекает и сохраняет векторные представления всех слов.
        
        Параметры:
            writer: опциональный объект SummaryWriter для логирования в TensorBoard.
        
        Возвращает:
            Словарь {слово: вектор_представления} для всех слов из словаря.
        """
        device = DEVICE
        model = TorchWord2Vec(len(self.vocab), self.vector_size).to(device)
        optimizer = optim.SGD(model.parameters(), lr=self.lr)

        dummy_centers = torch.randint(0, len(self.vocab), (BATCH_SIZE,)).to(device)
        dummy_contexts = torch.randint(0, len(self.vocab), (BATCH_SIZE,)).to(device)
        dummy_negatives = torch.randint(0, len(self.vocab), (BATCH_SIZE, self.negative)).to(device)
        
        # Capture model summary for TensorBoard
        summary_buffer = io.StringIO()
        old_stdout = sys.stdout
        sys.stdout = summary_buffer
        
        summary(
            model,
            input_data=(dummy_centers, dummy_contexts, dummy_negatives),
            col_names=["input_size", "output_size", "num_params", "trainable"],
            row_settings=["var_names"],
            verbose=1,
        )
        
        sys.stdout = old_stdout
        summary_text = summary_buffer.getvalue()
        
        # Log summary to TensorBoard
        if writer is not None:
            writer.add_text('Model/Summary', summary_text)
        
        # Print to console
        print(summary_text)

        self._build_training_pairs()

        total_words = sum(len([w for w in sent if w in self.vocab]) for sent in self.sentences)
        words_processed = 0

        for epoch in range(self.epochs):
            epoch_words = 0
            epoch_loss = 0
            batches = 0

            while epoch_words < total_words:
                lr = self.lr * (1 - words_processed / (self.epochs * total_words))
                for param_group in optimizer.param_groups:
                    param_group['lr'] = max(lr, 1e-4)
                
                centers, contexts, negatives = self._generate_train_batch()
                centers = centers.to(device)
                contexts = contexts.to(device)
                negatives = negatives.to(device)

                optimizer.zero_grad()
                loss = model(centers, contexts, negatives)
                loss.backward()
                optimizer.step()

                epoch_loss += loss
                epoch_words += len(centers)
                words_processed += len(centers)
                batches += 1
    
                
            avg_epoch_loss = epoch_loss / batches
            if writer is not None:
                writer.add_scalar('Loss/epoch/LR', avg_epoch_loss, epoch + 1, lr)
            
            print(f"Epoch {epoch+1}/{self.epochs} | Loss {avg_epoch_loss:.7f} | LR: {lr:.5f}")
            
        embeddings = model.embeddings.weight.cpu().detach().numpy()
        self.wv = {word: embeddings[index] for word, index in self.vocab.items()}
        return self.wv

In [57]:
train_ds = Dataset.from_json(f"{PATH_TO_CACHE}/train.jsonl")
test_ds = Dataset.from_json(f"{PATH_TO_CACHE}/test.jsonl")


def tokenize(text):
    return re.findall(r"\b[a-z]{2,}\b", text.lower())


tokenized_texts = [tokenize(text) for text in train_ds["text"]]

# Build vocab
all_tokens = [token for tokens in tokenized_texts for token in tokens]
vocab_freq = Counter(all_tokens)
vocab = {w for w, f in vocab_freq.items() if f >= 10}

# Filter tokens (reuse tokenized_texts)
sentences = [[t for t in tokens if t in vocab] for tokens in tokenized_texts]

trainer = Word2VecTrainer(
    sentences,
    embedding_dim=EMBEDDINGS_DIM,
    batch_size=BATCH_SIZE,
    window=WINDOW,
    min_count=MIN_COUNT,
    negative=NEGATIVE,
    sample_thr=SAMPLE,
    epochs=EPOCHS,
    lr=LEARNING_RATE)

trainer.train(writer=writer_tb)  # Pass TensorBoard writer for logging

model = trainer

trainer.wv['good']

Vocab size: 19792
Layer (type (var_name))                  Input Shape               Output Shape              Param #                   Trainable
TorchWord2Vec (TorchWord2Vec)            [128]                     --                        --                        True
├─Embedding (embeddings)                 [128]                     [128, 100]                1,979,200                 True
├─Embedding (output_weights)             [128]                     [128, 100]                1,979,200                 True
├─Embedding (output_weights)             [128, 5]                  [128, 5, 100]             (recursive)               True
Total params: 3,958,400
Trainable params: 3,958,400
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 760.01
Input size (MB): 0.01
Forward/backward pass size (MB): 0.72
Params size (MB): 15.83
Estimated Total Size (MB): 16.56

Generated 22,198,193 training pairs
Epoch 1/10 | Loss 4.1573930 | LR: 0.02250
Epoch 2/10 | Loss 4.1528707 | LR: 0.02000
E

array([ 1.10861674e-01, -1.73326448e-01, -9.78695154e-02,  2.43241191e-02,
        3.64865839e-01, -8.40029269e-02,  4.89792824e-02, -1.33325225e-02,
        1.13043398e-01, -7.60658011e-02, -1.09517880e-01, -1.48802161e-01,
        2.33037379e-02,  1.19545631e-01,  8.64112675e-02, -1.46701127e-01,
        1.72444865e-01,  8.59323740e-02, -1.06731668e-01,  5.35452180e-02,
        3.38158198e-02,  1.86770689e-02, -6.78312629e-02,  2.61240136e-02,
       -1.01415768e-01,  4.84500714e-02, -9.22616571e-02, -2.10567359e-02,
        2.26559848e-01,  1.22150302e-01, -7.56092966e-02,  3.42380046e-03,
        2.34148074e-02,  3.04392338e-01, -9.91366282e-02, -9.05564576e-02,
        1.12828948e-01, -1.63293138e-01,  4.52697277e-02,  7.86870643e-02,
        3.64081860e-02, -1.83368385e-01,  4.88551557e-02,  1.14883855e-01,
        6.85226023e-02, -1.23422444e-01, -4.30901013e-02,  1.30883440e-01,
        1.90973580e-02, -4.16886695e-02,  4.68208864e-02, -1.17405273e-01,
        1.82364974e-02,  

#### Проверка точности word2vec на классификационной задаче

In [58]:
def doc_vector(model, tokens, vocab_set):
    vectors = [model.wv[w] for w in tokens if w in model.wv]
    return np.mean(vectors, axis=0) if vectors else np.zeros(model.vector_size)

# 2. Prepare train data
train_texts = train_ds["text"]
train_labels = train_ds["label"]
train_tokens = [tokenize(text) for text in train_texts]
X_train = np.array([doc_vector(model, tokens, vocab) for tokens in train_tokens])
y_train = np.array(train_labels)

# 3. Prepare test data
test_texts = test_ds["text"]
test_labels = test_ds["label"]
test_tokens = [tokenize(text) for text in test_texts]
X_test = np.array([doc_vector(model, tokens, vocab) for tokens in test_tokens])
y_test = np.array(test_labels)

# 4. Train classifier
clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, y_train)

# 5. Evaluate
y_pred_test = clf.predict(X_test)
accuracy = accuracy_score(y_test, y_pred_test)

y_pred_train = clf.predict(X_train)
train_accuracy = accuracy_score(y_train, y_pred_train)
print(f"IMDB Sentiment Accuracy: {accuracy:.4f}")
print(f"IMDB Sentiment Train Accuracy: {train_accuracy:.4f}")

IMDB Sentiment Accuracy: 0.5225
IMDB Sentiment Train Accuracy: 0.5369


In [59]:
with open(f'{PATH_TO_MODEL}\\word2vec_embeddings.pkl', 'wb') as f:
    pickle.dump(trainer.wv, f)


In [60]:
with open(f'{PATH_TO_MODEL}\\word2vec_embeddings.pkl', 'rb') as f:
    loaded_wv = pickle.load(f)

In [63]:
loaded_wv['good']

array([ 1.10861674e-01, -1.73326448e-01, -9.78695154e-02,  2.43241191e-02,
        3.64865839e-01, -8.40029269e-02,  4.89792824e-02, -1.33325225e-02,
        1.13043398e-01, -7.60658011e-02, -1.09517880e-01, -1.48802161e-01,
        2.33037379e-02,  1.19545631e-01,  8.64112675e-02, -1.46701127e-01,
        1.72444865e-01,  8.59323740e-02, -1.06731668e-01,  5.35452180e-02,
        3.38158198e-02,  1.86770689e-02, -6.78312629e-02,  2.61240136e-02,
       -1.01415768e-01,  4.84500714e-02, -9.22616571e-02, -2.10567359e-02,
        2.26559848e-01,  1.22150302e-01, -7.56092966e-02,  3.42380046e-03,
        2.34148074e-02,  3.04392338e-01, -9.91366282e-02, -9.05564576e-02,
        1.12828948e-01, -1.63293138e-01,  4.52697277e-02,  7.86870643e-02,
        3.64081860e-02, -1.83368385e-01,  4.88551557e-02,  1.14883855e-01,
        6.85226023e-02, -1.23422444e-01, -4.30901013e-02,  1.30883440e-01,
        1.90973580e-02, -4.16886695e-02,  4.68208864e-02, -1.17405273e-01,
        1.82364974e-02,  

### Выводы


- Не вся информация сохраняется в TensorBoard, но основное там есть, модель недообучена, но все равно какиет-то результаты выдает
- В целом была построена word2vec skip-gram модель, какие-то веса словам она дала, эти веса были протестированы в построении логистической регрессии для классификации текста
- В ходе обучения Loss спустился с 4.1 до 3.6 за 10 эпох, точность лог. рег. достигла 0.52 на тестовой выборке и 0.53 на тренировочной
- Модель недообучена, возможно слишком маленький батч, недостаточно эпох, нужно попробовать другие оптимизаторы, увеличить размерность пространства, поменять другие гиперпараметры(но остальные не должны особо изменить картину)